# Tlemcen-NeuroOncMRI — Pipeline complet et propre
## PBT vs CM : Q1 (fuite de données), Q2 (fusion image+texte), Q3 (GAT)

Ce notebook consolide toute l'expérimentation validée, sans les versions
intermédiaires buguées. Structure :
1. Config & vérification des chemins
2. Chargement des labels validés + texte
3. Q1 — Modèle texte + analyse d'ablation (fuite de données)
4. Q2a — Modèle image (embeddings gelés, approche stable)
5. Q3 — GAT (similarité inter-patients)
6. Q2b — Fusion (moyenne simple / pondérée / stacking)
7. Stabilité multi-seed
8. Tableau récapitulatif final


## 1. Config, GPU, imports

**Cette cellule doit être exécutée en premier**, avant tout autre import TensorFlow, pour que le paramétrage GPU prenne effet.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # force un seul GPU (evite l'instabilite multi-GPU)

import re
import glob
import random
import warnings
import time
import gc
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
)

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

gpus = tf.config.experimental.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"{len(gpus)} GPU(s) actif(s), memory growth active")


class Config:
    EXCEL_PATH = Path("/kaggle/input/datasets/souaadrahmoun/datasetcac/ClasseurPFE1 (1).xlsx")
    IMAGE_DIR = Path("/kaggle/input/datasets/souaadrahmoun/datacacfinal/images_v2_final/data p_s")
    IMG_SIZE = 224
    N_BOOTSTRAP = 2000
    MAX_IMGS_PER_PATIENT = 20  # plafond : evite que les patients a 150+ images dominent


cfg = Config()

for p in [cfg.EXCEL_PATH, cfg.IMAGE_DIR]:
    print(p, "->", p.exists())

## 2. Chargement des donnees (label officiel valide + texte complet)

In [ ]:
df = pd.read_excel(cfg.EXCEL_PATH, sheet_name="Feuil1")
for col in ["Origine_Tumeur", "Type_General", "Sexe", "Localisation",
            "Metastase_Origine", "ANNOTATION", "Rapport"]:
    # Gere les eventuelles variantes d'accents dans les noms de colonnes
    matching = [c for c in df.columns if c.replace("é", "e").replace("è", "e") == col.replace("é","e")]
    real_col = matching[0] if matching else col
    if real_col in df.columns:
        df[real_col] = df[real_col].astype(str).fillna("").replace("nan", "")

# Normalisation des noms de colonnes accentues (Origine_Tumeur, Type_Général...)
df.columns = [c for c in df.columns]

y = (df["Origine_Tumeur"].astype(str).str.strip() == "Secondaire").astype(int).values
patient_ids = df["ID_Patient"].values

print(f"Patients : {len(y)}  (CP={sum(y==0)}, CS={sum(y==1)})")
assert len(y) == 45, "Attendu 45 patients"

## 3. Q1 — Modele texte + analyse d'ablation (fuite de donnees)

**Hypothese H1** : un modele texte+radio SANS variable clinique triviale
performe nettement moins bien qu'avec — quantifie l'ampleur d'une fuite
potentielle. Toutes les etapes (TF-IDF, selection de features, scaling)
sont refit a l'interieur de chaque fold LOO-CV.

Note de reproductibilite : `mutual_info_classif` a sa propre composante
aleatoire interne, explicitement seedee ici (bug trouve et corrige en
cours de route — sans ce correctif, deux runs "identiques" pouvaient
donner des AUC differant de 0.10+).

In [ ]:
CLASS_REVEALING_TERMS = [
    r"\bprimaire\b", r"\bsecondaire\b", r"m[ée]tastase\w*", r"m[ée]tastatique\w*",
]

def mask_class_revealing_terms(text_series):
    out = text_series.astype(str)
    for pat in CLASS_REVEALING_TERMS:
        out = out.str.replace(pat, " CLASSE ", regex=True, flags=re.IGNORECASE)
    return out


# Colonnes categorielles traitees a part : encodees a l'INTERIEUR de la boucle
# LOO-CV (fit sur train uniquement), pas globalement -> rigueur totale, meme
# si l'impact reel etait deja quasi nul (LabelEncoder n'utilise pas y).
CATEGORICAL_COLS = ["Sexe_raw", "Localisation_raw"]

def build_safe_manual_features(df):
    """Features radiologiques/demographiques SANS variable clinique triviale.
    Sexe et Localisation restent en texte brut ici (encodage differe a
    l'interieur du LOO-CV, voir run_loocv_text_model)."""
    feat = pd.DataFrame(index=df.index)
    feat["Age"] = pd.to_numeric(df["Âge"] if "Âge" in df.columns else df["Age"], errors="coerce")
    feat["Sexe_raw"] = df["Sexe"].astype(str)
    feat["Localisation_raw"] = df["Localisation"].astype(str)

    annot_col = "ANNOTATION" if "ANNOTATION" in df.columns else df.columns[df.columns.str.contains("ANNOT", case=False)][0]
    annot = mask_class_revealing_terms(df[annot_col])
    nlp_patterns = {
        "nlp_necrose": r"n[ée]cros", "nlp_oedeme": r"[oœ]d[eè]me",
        "nlp_effet_masse": r"effet de masse", "nlp_rehaussement": r"rehaussement",
        "nlp_saignement": r"saignement|h[ée]morrag", "nlp_multiple": r"multiple|plusieurs",
        "nlp_unique": r"\bunique\b", "nlp_engagement": r"engagement|sous-falc",
        "nlp_spectro": r"spectroscop", "nlp_annulaire": r"annulaire",
        "nlp_corps_calleux": r"corps calleux", "nlp_jonction_cortico": r"jonction cortico|cortico-sous-cortical",
        "nlp_infiltrant": r"infiltrant\w*",
    }
    annot_lower = annot.str.lower()
    for name, pat in nlp_patterns.items():
        feat[name] = annot_lower.str.contains(pat, na=False).astype(int)
    return feat


def build_leaky_features(df):
    """Features cliniques A RISQUE — isolees pour l'analyse d'ablation."""
    feat = pd.DataFrame(index=df.index)
    meta_col = "Métastase_Origine" if "Métastase_Origine" in df.columns else "Metastase_Origine"
    meta_vide = ["", "nan", "none", "non applicable", "non précisée", "-", "n/a"]
    feat["a_metastase"] = (~df[meta_col].astype(str).str.strip().str.lower().isin(meta_vide)).astype(int)
    return feat


class DenseTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X.toarray() if hasattr(X, "toarray") else X


def run_loocv_text_model(manual_feat, text_raw, y, k_best=20, seed=SEED):
    from sklearn.preprocessing import OrdinalEncoder

    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)

    cat_cols = [c for c in CATEGORICAL_COLS if c in manual_feat.columns]
    num_cols = [c for c in manual_feat.columns if c not in cat_cols]

    num_arr = manual_feat[num_cols].values.astype(float)
    cat_arr = manual_feat[cat_cols].values if cat_cols else None
    text_arr = text_raw.values

    for train_idx, test_idx in loo.split(num_arr):
        try:
            tfidf = TfidfVectorizer(max_features=150, ngram_range=(1, 2), min_df=2, max_df=0.90, sublinear_tf=True)
            X_tfidf_train = tfidf.fit_transform(text_arr[train_idx])
        except ValueError:
            tfidf = TfidfVectorizer(max_features=150, ngram_range=(1, 1), min_df=1, max_df=1.0, sublinear_tf=True)
            X_tfidf_train = tfidf.fit_transform(text_arr[train_idx])
        X_tfidf_test = tfidf.transform(text_arr[test_idx])

        dense = DenseTransformer()
        X_tfidf_train = dense.transform(X_tfidf_train)
        X_tfidf_test = dense.transform(X_tfidf_test)

        med = np.nanmedian(num_arr[train_idx], axis=0)
        Xm_train = np.where(np.isnan(num_arr[train_idx]), med, num_arr[train_idx])
        Xm_test = np.where(np.isnan(num_arr[test_idx]), med, num_arr[test_idx])

        # Encodage categoriel FIT SUR TRAIN UNIQUEMENT, categories inconnues
        # en test geree proprement (unknown_value=-1) plutot que de planter.
        if cat_arr is not None:
            enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            cat_train = enc.fit_transform(cat_arr[train_idx])
            cat_test = enc.transform(cat_arr[test_idx])
            Xm_train = np.hstack([Xm_train, cat_train])
            Xm_test = np.hstack([Xm_test, cat_test])

        X_train_full = np.hstack([Xm_train, X_tfidf_train])
        X_test_full = np.hstack([Xm_test, X_tfidf_test])

        scaler = StandardScaler()
        X_train_full = scaler.fit_transform(X_train_full)
        X_test_full = scaler.transform(X_test_full)

        mi_scorer = partial(mutual_info_classif, random_state=seed)
        k = min(k_best, X_train_full.shape[1])
        selector = SelectKBest(mi_scorer, k=k)
        X_train_sel = selector.fit_transform(X_train_full, y[train_idx])
        X_test_sel = selector.transform(X_test_full)

        clf = RandomForestClassifier(n_estimators=300, max_depth=5, class_weight="balanced", random_state=seed)
        clf.fit(X_train_sel, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test_sel)[:, 1]

    return oof_proba

### Metriques (utilisees dans tout le notebook)

In [ ]:
def summarize(y, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {
        "AUC": roc_auc_score(y, proba),
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0, average="macro"),
    }


def bootstrap_ci(y, proba, n_boot=2000, seed=SEED):
    rng = np.random.RandomState(seed)
    n = len(y)
    aucs = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y[idx], proba[idx]))
    return np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)

### Execution Q1 : texte SANS vs AVEC feature a risque

In [ ]:
annot_col = "ANNOTATION"
rapport_col = "Rapport"
text_raw = mask_class_revealing_terms(df[annot_col] + " " + df[rapport_col])

safe_feat = build_safe_manual_features(df)
leaky_feat = build_leaky_features(df)

print("=" * 70)
print("MODELE 1 - Texte+Radio SANS feature a risque")
print("=" * 70)
proba_text_safe = run_loocv_text_model(safe_feat, text_raw, y, seed=SEED)
res_text_safe = summarize(y, proba_text_safe)
res_text_safe["AUC_CI95"] = bootstrap_ci(y, proba_text_safe, cfg.N_BOOTSTRAP, seed=SEED)
print(res_text_safe)

print()
print("=" * 70)
print("MODELE 2 - Texte+Radio+Clinique AVEC feature a risque (ablation)")
print("=" * 70)
combined_feat = pd.concat([safe_feat, leaky_feat], axis=1)
proba_text_leaky = run_loocv_text_model(combined_feat, text_raw, y, seed=SEED)
res_text_leaky = summarize(y, proba_text_leaky)
res_text_leaky["AUC_CI95"] = bootstrap_ci(y, proba_text_leaky, cfg.N_BOOTSTRAP, seed=SEED)
print(res_text_leaky)

print()
delta_auc = res_text_leaky["AUC"] - res_text_safe["AUC"]
print(f"Delta AUC attribuable a la feature a risque : {delta_auc:+.4f}")

## 4. Q2a — Modele image (embeddings geles, approche stable)

**Choix methodologique important** : apres plusieurs echecs de fine-tuning
CNN (crashs memoire GPU repetes, instabilite due au nombre variable
d'images par patient), on utilise ici un extracteur **gele** (EfficientNetB0
pre-entraine ImageNet, aucun poids reentraine). Un seul passage d'extraction
(pas de boucle LOO-CV couteuse), puis un classifieur leger refit par fold —
beaucoup plus stable et rapide.

Les embeddings d'un patient sont **moyennes** -> un seul vecteur par patient,
ce qui regle structurellement le desequilibre du nombre d'images (5 a 155
selon les patients).

In [ ]:
def list_patient_images(patient_id, image_dir):
    pid_str = f"P{patient_id:02d}"
    files = []
    for cls in ["CP", "CS"]:
        cls_dir = image_dir / cls
        if cls_dir.exists():
            files += sorted(glob.glob(str(cls_dir / f"{pid_str} (*).jpg")))
            files += sorted(glob.glob(str(cls_dir / f"{pid_str} (*).png")))
    return files


def load_img(path, img_size=224):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=1, expand_animations=False)
    img = tf.image.grayscale_to_rgb(img)
    img = tf.image.resize(img, (img_size, img_size))
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img.numpy()


# Verification couverture
total_imgs = sum(len(list_patient_images(int(pid), cfg.IMAGE_DIR)) for pid in patient_ids)
print(f"Images retrouvees : {total_imgs} (attendu 2393)")

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

extractor = EfficientNetB0(include_top=False, weights="imagenet", pooling="avg")
extractor.trainable = False

rng = random.Random(SEED)
patient_embeddings = []
for pid in patient_ids:
    files = list_patient_images(int(pid), cfg.IMAGE_DIR)
    if len(files) > cfg.MAX_IMGS_PER_PATIENT:
        files = rng.sample(files, cfg.MAX_IMGS_PER_PATIENT)
    imgs = np.array([load_img(f, cfg.IMG_SIZE) for f in files])
    feats = extractor.predict(imgs, batch_size=8, verbose=0)
    patient_embeddings.append(feats.mean(axis=0))
    print(f"Patient {pid}: {len(files)} images -> embedding extrait")

patient_embeddings = np.stack(patient_embeddings)
np.save("/kaggle/working/effnet_embeddings.npy", patient_embeddings)
print("Embeddings extraits, shape:", patient_embeddings.shape)

In [ ]:
def run_loocv_on_embeddings(embeddings, y, seed=SEED):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    for train_idx, test_idx in loo.split(embeddings):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(embeddings[train_idx])
        X_test = scaler.transform(embeddings[test_idx])
        clf = RandomForestClassifier(n_estimators=300, max_depth=5, class_weight="balanced", random_state=seed)
        clf.fit(X_train, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test)[:, 1]
    return oof_proba


proba_image = run_loocv_on_embeddings(patient_embeddings, y, seed=SEED)
res_image = summarize(y, proba_image)
res_image["AUC_CI95"] = bootstrap_ci(y, proba_image, cfg.N_BOOTSTRAP, seed=SEED)
print(res_image)

pd.DataFrame({"ID_Patient": patient_ids, "y_true": y, "proba_image": proba_image}) \
    .to_csv("/kaggle/working/oof_image_effnet_frozen.csv", index=False)
print("Exporte -> oof_image_effnet_frozen.csv")

## 5. Q3 — GAT (similarite inter-patients)

**Cadrage methodologique** : graphe transductif (k-NN cosinus sur features
standardisees, construites sur l'ensemble des 45 patients), mais le label du
patient teste n'est **jamais** utilise pendant l'entrainement (masquage
strict a chaque fold). Resultat exploratoire, pas le modele principal.

**Attention instabilite documentee** : ce GAT s'est montre tres sensible a
la seed lors de nos tests (AUC variant de 0.31 a 0.65 selon la seed, moyenne
~0.42 sur 6 seeds — en dessous du hasard). A interpreter avec prudence,
potentiellement executer plusieurs seeds avant de conclure.

Necessite `pip install torch` si non deja present sur l'environnement Kaggle.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity

torch.manual_seed(SEED)


def build_patient_graph(features, k=5):
    n = features.shape[0]
    sim = cosine_similarity(features)
    np.fill_diagonal(sim, -np.inf)
    adjacency = np.zeros((n, n), dtype=bool)
    for i in range(n):
        neighbors = np.argsort(-sim[i])[:k]
        adjacency[i, neighbors] = True
    adjacency = adjacency | adjacency.T
    np.fill_diagonal(adjacency, True)
    return adjacency


class SimpleGATLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a = nn.Linear(2 * out_dim, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, x, adjacency):
        n = x.shape[0]
        h = self.W(x)
        h_i = h.unsqueeze(1).expand(n, n, -1)
        h_j = h.unsqueeze(0).expand(n, n, -1)
        e = self.leaky_relu(self.a(torch.cat([h_i, h_j], dim=-1)).squeeze(-1))
        e = e.masked_fill(~adjacency, float("-inf"))
        alpha = F.softmax(e, dim=1)
        return torch.matmul(alpha, h), alpha


class PatientGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim=16):
        super().__init__()
        self.gat1 = SimpleGATLayer(in_dim, hidden_dim)
        self.gat2 = SimpleGATLayer(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(self, x, adjacency):
        h, _ = self.gat1(x, adjacency)
        h = F.elu(h)
        h, alpha2 = self.gat2(h, adjacency)
        h = F.elu(h)
        return self.classifier(h), alpha2


def run_loocv_gat(features, y, k=5, hidden_dim=16, epochs=200, lr=0.01, seed=SEED):
    n = len(y)
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    adjacency = torch.tensor(build_patient_graph(features_scaled, k=k), dtype=torch.bool)
    x = torch.tensor(features_scaled, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)

    oof_proba = np.zeros(n)
    for test_idx in range(n):
        torch.manual_seed(seed)
        model = PatientGAT(in_dim=features.shape[1], hidden_dim=hidden_dim)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
        train_mask = torch.ones(n, dtype=torch.bool)
        train_mask[test_idx] = False
        class_counts = torch.bincount(y_t[train_mask])
        class_weights = 1.0 / class_counts.float()
        class_weights = class_weights / class_weights.sum() * 2

        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            logits, _ = model(x, adjacency)
            loss = F.cross_entropy(logits[train_mask], y_t[train_mask], weight=class_weights)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            logits, _ = model(x, adjacency)
            proba = F.softmax(logits, dim=1)[:, 1]
            oof_proba[test_idx] = proba[test_idx].item()

    return oof_proba


# Encodage des categorielles pour le graphe GAT (coherent avec le cadrage
# transductif deja documente : le graphe voit les features de tous les
# patients pour calculer les voisins, seul le label reste masque par fold)
from sklearn.preprocessing import OrdinalEncoder

safe_feat_encoded = safe_feat.copy()
cat_cols_gat = [c for c in safe_feat.columns if c.endswith("_raw")]
if cat_cols_gat:
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    safe_feat_encoded[cat_cols_gat] = enc.fit_transform(safe_feat[cat_cols_gat])

# Graphe texte seul
proba_gat_text = run_loocv_gat(safe_feat_encoded.values.astype(float), y, k=5, seed=SEED)
print("GAT (texte seul):", summarize(y, proba_gat_text))

# Graphe texte + image combines
combined_features = np.hstack([safe_feat_encoded.values.astype(float), patient_embeddings])
proba_gat_combined = run_loocv_gat(combined_features, y, k=5, seed=SEED)
print("GAT (texte+image):", summarize(y, proba_gat_combined))

## 6. Q2b — Fusion (moyenne simple / ponderee / stacking)

Toute ponderation apprise l'est **en LOO-CV**, jamais sur l'ensemble complet
(contrairement a la "Super Fusion" du notebook original, qui optimisait
seuil et poids directement sur les 45 patients — biais optimiste).

In [ ]:
def fusion_moyenne_simple(branch_probas):
    stacked = np.stack(list(branch_probas.values()), axis=1)
    return stacked.mean(axis=1)


def fusion_pondere_loocv(branch_probas, y):
    names = list(branch_probas.keys())
    n_branches = len(names)
    stacked = np.stack([branch_probas[k] for k in names], axis=1)
    n = len(y)
    oof = np.zeros(n)

    if n_branches == 2:
        grid = [(w, 1 - w) for w in np.linspace(0, 1, 21)]
    else:
        rng = np.random.RandomState(42)
        grid = [tuple(r) for r in rng.dirichlet(np.ones(n_branches), size=200)]

    loo = LeaveOneOut()
    for train_idx, test_idx in loo.split(stacked):
        best_w, best_auc = None, -1
        for w in grid:
            w = np.array(w)
            if len(np.unique(y[train_idx])) < 2:
                continue
            auc = roc_auc_score(y[train_idx], stacked[train_idx] @ w)
            if auc > best_auc:
                best_auc, best_w = auc, w
        oof[test_idx] = stacked[test_idx] @ best_w
    return oof


def fusion_stacking_loocv(branch_probas, y):
    names = list(branch_probas.keys())
    stacked = np.stack([branch_probas[k] for k in names], axis=1)
    n = len(y)
    oof = np.zeros(n)
    loo = LeaveOneOut()
    for train_idx, test_idx in loo.split(stacked):
        meta = LogisticRegression(class_weight="balanced", max_iter=1000)
        meta.fit(stacked[train_idx], y[train_idx])
        oof[test_idx] = meta.predict_proba(stacked[test_idx])[:, 1]
    return oof


branches = {
    "texte": proba_text_safe,
    "image": proba_image,
    "gat": proba_gat_combined,
}

fusion_results = {}
for nom, fn in [
    ("Moyenne simple", lambda: fusion_moyenne_simple(branches)),
    ("Moyenne ponderee (LOO-CV)", lambda: fusion_pondere_loocv(branches, y)),
    ("Stacking logistique (LOO-CV)", lambda: fusion_stacking_loocv(branches, y)),
]:
    proba = fn()
    res = summarize(y, proba)
    res["AUC_CI95"] = bootstrap_ci(y, proba, cfg.N_BOOTSTRAP, seed=SEED)
    fusion_results[nom] = res
    print(f"--- {nom} ---")
    print(res)
    print()

## 7. Stabilite multi-seed (texte — le plus rapide a re-tester)

**Rappel important decouvert en cours de route** : un seul seed peut donner
un resultat trompeur (ex: le GAT variait de 0.31 a 0.65 selon la seed). On
verifie ici la stabilite du modele texte sur plusieurs seeds avant de figer
le chiffre "officiel" a rapporter dans l'article.

In [ ]:
seeds_to_test = [1, 2, 3, 4, 42, 777]
multiseed_results = {}
for s in seeds_to_test:
    proba_s = run_loocv_text_model(safe_feat, text_raw, y, seed=s)
    auc_s = summarize(y, proba_s)["AUC"]
    multiseed_results[s] = auc_s
    print(f"seed={s}: AUC={auc_s:.4f}")

aucs = np.array(list(multiseed_results.values()))
print()
print(f"AUC moyenne multi-seed : {aucs.mean():.4f} +/- {aucs.std():.4f}")
print(f"Min-Max : [{aucs.min():.4f}, {aucs.max():.4f}]")
print()
print("=> C'est cette moyenne multi-seed, pas un seed isole, qui doit etre")
print("   rapportee comme resultat 'officiel' dans l'article.")

## 8. Tableau recapitulatif final

In [ ]:
final_summary = pd.DataFrame({
    "Texte seul (safe)": res_text_safe,
    "Texte + feature a risque": res_text_leaky,
    "Image seule (embeddings geles)": res_image,
    "GAT (texte+image)": summarize(y, proba_gat_combined),
    **fusion_results,
}).T

print(final_summary.to_string())
final_summary.to_csv("/kaggle/working/resultats_finaux_complets.csv")
print()
print("Sauvegarde -> resultats_finaux_complets.csv")

# Sauvegarde de toutes les probas OOF individuelles, pour analyse ulterieure
oof_all = pd.DataFrame({
    "ID_Patient": patient_ids,
    "y_true": y,
    "proba_text_safe": proba_text_safe,
    "proba_text_leaky": proba_text_leaky,
    "proba_image": proba_image,
    "proba_gat_text": proba_gat_text,
    "proba_gat_combined": proba_gat_combined,
})
oof_all.to_csv("/kaggle/working/oof_all_branches.csv", index=False)
print("Sauvegarde -> oof_all_branches.csv")

## Notes methodologiques a garder pour la redaction de l'article

- **Q1 (fuite)** : resultat le plus solide. Delta AUC quantifie precisement
  l'impact d'une variable clinique triviale.
- **Q2 (fusion image+texte)** : l'image seule (meme avec embeddings foundation
  geles, agregation patient-level) reste proche du hasard sur cette cohorte —
  a documenter honnetement plutot que de chercher a "forcer" un meilleur chiffre.
- **Q3 (GAT)** : instable selon la seed — necessite un rapport multi-seed,
  jamais un seul chiffre isole.
- **Fuite image dans le notebook original** : le split train/test original
  melangeait des images du meme patient entre train et test (`split_dataset`
  operait au niveau image, pas patient) — explique l'ecart massif entre les
  AUC=0.92-0.93 originaux et nos resultats patient-level stricts (~0.5-0.6).
- **Reproductibilite** : `mutual_info_classif` necessite un `random_state`
  explicite, sinon deux runs "identiques" divergent significativement.


In [ ]:
import glob
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def list_patient_images(patient_id, image_dir):
    pid_str = f"P{patient_id:02d}"
    files = []
    for cls in ["CP", "CS"]:
        cls_dir = image_dir / cls
        if cls_dir.exists():
            files += sorted(glob.glob(str(cls_dir / f"{pid_str} (*).jpg")))
            files += sorted(glob.glob(str(cls_dir / f"{pid_str} (*).png")))
    return files


def extract_per_image_embeddings(patient_ids, image_dir, extractor, img_size=224,
                                  max_imgs_per_patient=30, seed=42):
    rng = random.Random(seed)
    embeddings_per_patient = {}
    for pid in patient_ids:
        files = list_patient_images(int(pid), image_dir)
        if len(files) > max_imgs_per_patient:
            files = rng.sample(files, max_imgs_per_patient)
        imgs = np.array([load_img(f, img_size) for f in files])
        feats = extractor.predict(imgs, batch_size=8, verbose=0)
        embeddings_per_patient[int(pid)] = feats
        print(f"Patient {pid}: {len(files)} embeddings slice-level extraits")
    return embeddings_per_patient


def run_loocv_slicelevel(patient_ids, y, embeds_per_patient, seed=42,
                          classifier="logreg", agg="mean"):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)

    for train_idx, test_idx in loo.split(patient_ids):
        train_pids = patient_ids[train_idx]
        test_pid = patient_ids[test_idx][0]
        y_train_patient = y[train_idx]

        X_train_slices, y_train_slices = [], []
        for pid, label in zip(train_pids, y_train_patient):
            embeds = embeds_per_patient[int(pid)]
            X_train_slices.append(embeds)
            y_train_slices.append(np.full(len(embeds), label))
        X_train_slices = np.vstack(X_train_slices)
        y_train_slices = np.concatenate(y_train_slices)

        scaler = StandardScaler()
        X_train_slices = scaler.fit_transform(X_train_slices)

        if classifier == "logreg":
            clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0, random_state=seed)
        else:
            from sklearn.ensemble import RandomForestClassifier
            clf = RandomForestClassifier(n_estimators=300, max_depth=5, class_weight="balanced", random_state=seed)
        clf.fit(X_train_slices, y_train_slices)

        X_test_slices = scaler.transform(embeds_per_patient[int(test_pid)])
        proba_slices = clf.predict_proba(X_test_slices)[:, 1]
        oof_proba[test_idx] = np.median(proba_slices) if agg == "median" else np.mean(proba_slices)

    return oof_proba


# --- Extraction (une seule fois) ---
embeds_per_image = extract_per_image_embeddings(
    patient_ids, cfg.IMAGE_DIR, extractor, img_size=cfg.IMG_SIZE,
    max_imgs_per_patient=30, seed=SEED
)

# --- LOO-CV slice-level ---
proba_image_slicelevel = run_loocv_slicelevel(
    patient_ids, y, embeds_per_image, seed=SEED, classifier="logreg", agg="mean"
)
res_slicelevel = summarize(y, proba_image_slicelevel)
res_slicelevel["AUC_CI95"] = bootstrap_ci(y, proba_image_slicelevel, seed=SEED)

print("=" * 70)
print("COMPARAISON : agregation embeddings (ancien) vs agregation probas (nouveau)")
print("=" * 70)
print("Ancien (embeddings moyennes avant classification) :", res_image)
print("Nouveau (classification slice-level + agregation probas) :", res_slicelevel)

pd.DataFrame({"ID_Patient": patient_ids, "y_true": y, "proba_image_slicelevel": proba_image_slicelevel}) \
    .to_csv("/kaggle/working/oof_image_slicelevel.csv", index=False)
print("\nExporte -> oof_image_slicelevel.csv")

In [ ]:
SEEDS_TO_TEST = [1, 2, 3, 4, 42, 777]
aucs_slicelevel = []
for s in SEEDS_TO_TEST:
    proba_s = run_loocv_slicelevel(patient_ids, y, embeds_per_image, seed=s, classifier="logreg", agg="mean")
    auc_s = summarize(y, proba_s)["AUC"]
    aucs_slicelevel.append(auc_s)
    print(f"seed={s}: AUC={auc_s:.4f}")

aucs_slicelevel = np.array(aucs_slicelevel)
print(f"\nAUC moyenne multi-seed (image slice-level) : {aucs_slicelevel.mean():.4f} +/- {aucs_slicelevel.std():.4f}")

In [ ]:
seeds_to_test = [1, 2, 3, 42, 777]
aucs_slicelevel = []
for s in seeds_to_test:
    embeds_s = extract_per_image_embeddings(
        patient_ids, cfg.IMAGE_DIR, extractor, img_size=cfg.IMG_SIZE,
        max_imgs_per_patient=30, seed=s
    )
    proba_s = run_loocv_slicelevel(patient_ids, y, embeds_s, seed=42, classifier="logreg", agg="mean")
    auc_s = summarize(y, proba_s)["AUC"]
    aucs_slicelevel.append(auc_s)
    print(f"seed extraction={s}: AUC={auc_s:.4f}")

print(f"\nMoyenne: {np.mean(aucs_slicelevel):.4f} +/- {np.std(aucs_slicelevel):.4f}")

In [ ]:
# --- Features image pour le GAT : moyenne des embeddings slice-level par patient ---
# (le GAT a besoin d'UN vecteur par patient, pas d'un ensemble par slice)
image_features_for_gat = np.stack([
    embeds_per_image[int(pid)].mean(axis=0) for pid in patient_ids
])

# --- Encodage des categorielles texte (comme deja fait precedemment) ---
from sklearn.preprocessing import OrdinalEncoder
safe_feat_encoded = safe_feat.copy()
cat_cols_gat = [c for c in safe_feat.columns if c.endswith("_raw")]
if cat_cols_gat:
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    safe_feat_encoded[cat_cols_gat] = enc.fit_transform(safe_feat[cat_cols_gat])

# --- Graphe combine texte + nouveaux embeddings image ---
combined_features_v2 = np.hstack([safe_feat_encoded.values.astype(float), image_features_for_gat])

seeds_to_test = [1, 2, 3, 4, 42, 777]
aucs_gat_v2 = []
for s in seeds_to_test:
    proba_s = run_loocv_gat(combined_features_v2, y, k=5, seed=s)
    auc_s = summarize(y, proba_s)["AUC"]
    aucs_gat_v2.append(auc_s)
    print(f"seed={s}: AUC={auc_s:.4f}")

aucs_gat_v2 = np.array(aucs_gat_v2)
print(f"\nGAT texte+image (embeddings slice-level) — moyenne multi-seed : {aucs_gat_v2.mean():.4f} +/- {aucs_gat_v2.std():.4f}")
print(f"Min-Max : [{aucs_gat_v2.min():.4f}, {aucs_gat_v2.max():.4f}]")

pd.DataFrame({"seed": seeds_to_test, "AUC": aucs_gat_v2}).to_csv("/kaggle/working/gat_v2_multiseed.csv", index=False)

In [ ]:
# Installation du foundation model (necessite internet Kaggle active)
!pip install open_clip_torch -q

import torch
import open_clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Chargement : BiomedCLIP en priorite, repli sur CLIP general si indisponible ---
try:
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-16",
        pretrained="hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
    model_name_used = "BiomedCLIP (specifique medical)"
except Exception as e:
    print(f"BiomedCLIP indisponible ({e}), repli sur CLIP general")
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    model_name_used = "CLIP general (ViT-B-32, OpenAI)"

model.eval().to(device)
print(f"Modele charge : {model_name_used}")


def extract_per_image_embeddings_foundation(patient_ids, image_dir, model, preprocess, device,
                                             max_imgs_per_patient=30, seed=42):
    rng = random.Random(seed)
    embeddings_per_patient = {}
    for pid in patient_ids:
        files = list_patient_images(int(pid), image_dir)
        if len(files) > max_imgs_per_patient:
            files = rng.sample(files, max_imgs_per_patient)

        imgs = torch.stack([preprocess(Image.open(f).convert("RGB")) for f in files]).to(device)
        with torch.no_grad():
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        embeddings_per_patient[int(pid)] = feats.cpu().numpy()
        print(f"Patient {pid}: {len(files)} embeddings {model_name_used.split()[0]} extraits")

    return embeddings_per_patient


# --- Extraction (une seule fois) ---
embeds_foundation = extract_per_image_embeddings_foundation(
    patient_ids, cfg.IMAGE_DIR, model, preprocess, device,
    max_imgs_per_patient=30, seed=SEED
)

# --- Classification slice-level (reutilise la fonction deja definie) ---
proba_foundation = run_loocv_slicelevel(
    patient_ids, y, embeds_foundation, seed=SEED, classifier="logreg", agg="mean"
)
res_foundation = summarize(y, proba_foundation)
res_foundation["AUC_CI95"] = bootstrap_ci(y, proba_foundation, seed=SEED)

print("=" * 70)
print(f"COMPARAISON DES EXTRACTEURS (classification slice-level identique)")
print("=" * 70)
print("EfficientNetB0 (ImageNet)      :", res_slicelevel)
print(f"{model_name_used:30s} :", res_foundation)

pd.DataFrame({"ID_Patient": patient_ids, "y_true": y, "proba_image_foundation": proba_foundation}) \
    .to_csv("/kaggle/working/oof_image_foundation_slicelevel.csv", index=False)
print("\nExporte -> oof_image_foundation_slicelevel.csv")

In [ ]:
import torch
import open_clip
from PIL import Image

device = "cpu"  # force CPU : evite l'incompatibilite CUDA/torch rencontree avec le GPU de cette session

try:
    model, _, preprocess = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
    model_name_used = "BiomedCLIP (specifique medical)"
except Exception as e:
    print(f"BiomedCLIP indisponible ({e}), repli sur CLIP general")
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    model_name_used = "CLIP general (ViT-B-32, OpenAI)"

model.eval().to(device)
print(f"Modele charge : {model_name_used} (sur {device})")


def extract_per_image_embeddings_foundation(patient_ids, image_dir, model, preprocess, device,
                                             max_imgs_per_patient=30, seed=42):
    rng = random.Random(seed)
    embeddings_per_patient = {}
    for pid in patient_ids:
        files = list_patient_images(int(pid), image_dir)
        if len(files) > max_imgs_per_patient:
            files = rng.sample(files, max_imgs_per_patient)

        imgs = torch.stack([preprocess(Image.open(f).convert("RGB")) for f in files]).to(device)
        with torch.no_grad():
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        embeddings_per_patient[int(pid)] = feats.cpu().numpy()
        print(f"Patient {pid}: {len(files)} embeddings extraits")

    return embeddings_per_patient


embeds_foundation = extract_per_image_embeddings_foundation(
    patient_ids, cfg.IMAGE_DIR, model, preprocess, device,
    max_imgs_per_patient=30, seed=SEED
)

proba_foundation = run_loocv_slicelevel(
    patient_ids, y, embeds_foundation, seed=SEED, classifier="logreg", agg="mean"
)
res_foundation = summarize(y, proba_foundation)
res_foundation["AUC_CI95"] = bootstrap_ci(y, proba_foundation, seed=SEED)

print("=" * 70)
print(f"COMPARAISON DES EXTRACTEURS")
print("=" * 70)
print("EfficientNetB0 (ImageNet, slice-level)         :", res_slicelevel)
print(f"{model_name_used:45s} :", res_foundation)

pd.DataFrame({"ID_Patient": patient_ids, "y_true": y, "proba_image_foundation": proba_foundation}) \
    .to_csv("/kaggle/working/oof_image_foundation_slicelevel.csv", index=False)
print("\nExporte -> oof_image_foundation_slicelevel.csv")

In [ ]:
!pip install transformers -q

import torch
from transformers import AutoTokenizer, AutoModel

device = "cpu"  # coherent avec le choix fait pour l'image (evite les soucis CUDA)

tokenizer = AutoTokenizer.from_pretrained("camembert-base")
camembert = AutoModel.from_pretrained("camembert-base").to(device)
camembert.eval()
print("CamemBERT charge")


def extract_camembert_embeddings(text_series, tokenizer, model, device, max_length=256):
    """Un embedding par document, moyenne des tokens (mean pooling), gele."""
    embeddings = []
    with torch.no_grad():
        for text in text_series:
            inputs = tokenizer(text, return_tensors="pt", truncation=True,
                                max_length=max_length, padding=True).to(device)
            outputs = model(**inputs)
            # Mean pooling sur les tokens (masque d'attention pour ignorer le padding)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            summed = (outputs.last_hidden_state * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            emb = (summed / counts).squeeze(0).cpu().numpy()
            embeddings.append(emb)
    return np.stack(embeddings)


# --- Extraction (une seule fois, PAS dans la boucle LOO-CV) ---
camembert_embeddings = extract_camembert_embeddings(text_raw, tokenizer, camembert, device)
print("Embeddings CamemBERT extraits, shape:", camembert_embeddings.shape)

# --- LOO-CV : classifieur leger sur ces embeddings (meme structure que l'image) ---
def run_loocv_on_text_embeddings(embeddings, y, seed=42):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    for train_idx, test_idx in loo.split(embeddings):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(embeddings[train_idx])
        X_test = scaler.transform(embeddings[test_idx])
        clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0, random_state=seed)
        clf.fit(X_train, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test)[:, 1]
    return oof_proba


proba_camembert = run_loocv_on_text_embeddings(camembert_embeddings, y, seed=SEED)
res_camembert = summarize(y, proba_camembert)
res_camembert["AUC_CI95"] = bootstrap_ci(y, proba_camembert, seed=SEED)

print("=" * 70)
print("COMPARAISON : TF-IDF+manuel (ancien) vs CamemBERT gele (nouveau)")
print("=" * 70)
print("TF-IDF + features manuelles :", res_text_safe)
print("CamemBERT (embeddings geles)  :", res_camembert)

In [ ]:
seeds_to_test = [1, 2, 3, 4, 42, 777]
aucs_biomedclip = []
for s in seeds_to_test:
    embeds_s = extract_per_image_embeddings_foundation(
        patient_ids, cfg.IMAGE_DIR, model, preprocess, device,
        max_imgs_per_patient=30, seed=s
    )
    proba_s = run_loocv_slicelevel(patient_ids, y, embeds_s, seed=42, classifier="logreg", agg="mean")
    auc_s = summarize(y, proba_s)["AUC"]
    aucs_biomedclip.append(auc_s)
    print(f"seed extraction={s}: AUC={auc_s:.4f}")

aucs_biomedclip = np.array(aucs_biomedclip)
print(f"\nBiomedCLIP multi-seed : {aucs_biomedclip.mean():.4f} +/- {aucs_biomedclip.std():.4f}")
print(f"Min-Max : [{aucs_biomedclip.min():.4f}, {aucs_biomedclip.max():.4f}]")

In [ ]:
seeds_to_test = [1, 2, 3, 4, 42, 777]
aucs_camembert = []
for s in seeds_to_test:
    proba_s = run_loocv_on_text_embeddings(camembert_embeddings, y, seed=s)
    auc_s = summarize(y, proba_s)["AUC"]
    aucs_camembert.append(auc_s)
    print(f"seed={s}: AUC={auc_s:.4f}")

aucs_camembert = np.array(aucs_camembert)
print(f"\nCamemBERT multi-seed : {aucs_camembert.mean():.4f} +/- {aucs_camembert.std():.4f}")

In [ ]:
# RadImageNet n'est pas toujours facile d'acces (poids parfois heberges hors HF) —
# repli automatique sur DINOv2 (self-supervise, tres robuste en generaliste) si indisponible
try:
    from huggingface_hub import hf_hub_download
    import tensorflow as tf
    # Tentative via un mirroir HF communautaire (le depot officiel distribue souvent par Google Drive)
    weights_path = hf_hub_download(repo_id="Lawhy/RadImageNet-ResNet50", filename="RadImageNet-ResNet50_notop.h5")
    from tensorflow.keras.applications import ResNet50
    radimagenet_extractor = ResNet50(include_top=False, weights=weights_path, pooling="avg",
                                       input_shape=(224, 224, 3))
    radimagenet_extractor.trainable = False
    model_name_used_img = "RadImageNet-ResNet50 (radiologie specifique)"
    USE_TF_FOR_RAD = True
except Exception as e:
    print(f"RadImageNet indisponible ({e}), repli sur DINOv2")
    import torch
    dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
    dino.eval().to("cpu")
    model_name_used_img = "DINOv2 (self-supervise, generaliste robuste)"
    USE_TF_FOR_RAD = False

print("Modele image utilise :", model_name_used_img)

In [ ]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device="cpu")

sbert_embeddings = st_model.encode(text_raw.tolist(), show_progress_bar=True)
print("Embeddings SBERT multilingue extraits, shape:", sbert_embeddings.shape)

proba_sbert = run_loocv_on_text_embeddings(sbert_embeddings, y, seed=SEED)
res_sbert = summarize(y, proba_sbert)
res_sbert["AUC_CI95"] = bootstrap_ci(y, proba_sbert, seed=SEED)

print("=" * 70)
print("COMPARAISON TEXTE : TF-IDF vs CamemBERT vs SBERT multilingue")
print("=" * 70)
print("TF-IDF + manuel        :", res_text_safe)
print("CamemBERT (mean-pool)  :", res_camembert)
print("SBERT multilingue      :", res_sbert)

In [ ]:
def extract_dinov2_embeddings(patient_ids, image_dir, model, device, max_imgs_per_patient=30, seed=42):
    from torchvision import transforms
    from PIL import Image
    import random

    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    rng = random.Random(seed)
    embeddings_per_patient = {}
    for pid in patient_ids:
        files = list_patient_images(int(pid), image_dir)
        if len(files) > max_imgs_per_patient:
            files = rng.sample(files, max_imgs_per_patient)

        imgs = torch.stack([preprocess(Image.open(f).convert("RGB")) for f in files]).to(device)
        with torch.no_grad():
            feats = model(imgs)  # DINOv2 renvoie directement les embeddings CLS
        embeddings_per_patient[int(pid)] = feats.cpu().numpy()
        print(f"Patient {pid}: {len(files)} embeddings DINOv2 extraits")

    return embeddings_per_patient


embeds_dinov2 = extract_dinov2_embeddings(patient_ids, cfg.IMAGE_DIR, dino, "cpu", max_imgs_per_patient=30, seed=SEED)

proba_dinov2 = run_loocv_slicelevel(patient_ids, y, embeds_dinov2, seed=SEED, classifier="logreg", agg="mean")
res_dinov2 = summarize(y, proba_dinov2)
res_dinov2["AUC_CI95"] = bootstrap_ci(y, proba_dinov2, seed=SEED)

print("Comparaison finale des extracteurs image :")
print("EfficientNetB0 (ImageNet)     :", res_slicelevel)
print("BiomedCLIP (medical)          :", res_foundation)
print("DINOv2 (self-supervise)       :", res_dinov2)

In [ ]:
branches_v3 = {
    "texte_camembert": proba_camembert,
    "image_biomedclip": proba_foundation,
}

for nom, fn in [
    ("Moyenne simple", lambda: fusion_moyenne_simple(branches_v3)),
    ("Moyenne ponderee (LOO-CV)", lambda: fusion_pondere_loocv(branches_v3, y)),
    ("Stacking logistique (LOO-CV)", lambda: fusion_stacking_loocv(branches_v3, y)),
]:
    proba = fn()
    res = summarize(y, proba)
    res["AUC_CI95"] = bootstrap_ci(y, proba, seed=SEED)
    print(f"--- {nom} ---")
    print(res)
    print()

In [ ]:
from sklearn.decomposition import PCA
from scipy.stats import zscore

# ============================================================
# Rappel des branches individuelles
# ============================================================
print("Texte (CamemBERT)  :", summarize(y, proba_camembert))
print("Image (BiomedCLIP) :", summarize(y, proba_foundation))
print()

branches_v3 = {
    "texte_camembert": proba_camembert,
    "image_biomedclip": proba_foundation,
}

all_fusion_results = {}

# ============================================================
# 1-3. Fusion tardive (moyenne simple / ponderee / stacking)
# ============================================================
for nom, fn in [
    ("1. Moyenne simple", lambda: fusion_moyenne_simple(branches_v3)),
    ("2. Moyenne ponderee (LOO-CV)", lambda: fusion_pondere_loocv(branches_v3, y)),
    ("3. Stacking logistique (LOO-CV)", lambda: fusion_stacking_loocv(branches_v3, y)),
]:
    proba = fn()
    res = summarize(y, proba)
    res["AUC_CI95"] = bootstrap_ci(y, proba, seed=SEED)
    all_fusion_results[nom] = res

# ============================================================
# 4. Fusion precoce (concatenation + PCA + un seul classifieur)
# ============================================================
def run_loocv_early_fusion(text_embeddings, image_embeddings_per_patient, y,
                            n_components_text=10, n_components_image=10, seed=42):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    image_patient_level = np.stack([
        image_embeddings_per_patient[int(pid)].mean(axis=0) for pid in patient_ids
    ])
    for train_idx, test_idx in loo.split(text_embeddings):
        pca_text = PCA(n_components=n_components_text, random_state=seed)
        text_train = pca_text.fit_transform(text_embeddings[train_idx])
        text_test = pca_text.transform(text_embeddings[test_idx])
        pca_img = PCA(n_components=n_components_image, random_state=seed)
        img_train = pca_img.fit_transform(image_patient_level[train_idx])
        img_test = pca_img.transform(image_patient_level[test_idx])
        X_train = np.hstack([text_train, img_train])
        X_test = np.hstack([text_test, img_test])
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=0.5, random_state=seed)
        clf.fit(X_train, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test)[:, 1]
    return oof_proba

proba_early = run_loocv_early_fusion(camembert_embeddings, embeds_foundation, y, seed=SEED)
res_early = summarize(y, proba_early)
res_early["AUC_CI95"] = bootstrap_ci(y, proba_early, seed=SEED)
all_fusion_results["4. Fusion precoce (concat+PCA)"] = res_early

# ============================================================
# 5. Fusion guidee (le texte module la confiance accordee a l'image)
# ============================================================
def run_guided_fusion(proba_text, proba_image, y):
    text_confidence = 1 / (1 + np.exp(-zscore(proba_text)))
    agreement = 1 - np.abs(text_confidence - proba_image)
    w_text = agreement / (agreement + (1 - agreement) + 1e-9)
    return w_text * proba_text + (1 - w_text) * proba_image

proba_guided = run_guided_fusion(proba_camembert, proba_foundation, y)
res_guided = summarize(y, proba_guided)
res_guided["AUC_CI95"] = bootstrap_ci(y, proba_guided, seed=SEED)
all_fusion_results["5. Fusion guidee (texte module image)"] = res_guided

# ============================================================
# TABLEAU FINAL
# ============================================================
print("=" * 90)
print("TABLEAU COMPLET DES 5 STRATEGIES DE FUSION (branches : CamemBERT + BiomedCLIP)")
print("=" * 90)
fusion_df = pd.DataFrame(all_fusion_results).T
print(fusion_df.to_string())

best = fusion_df["AUC"].idxmax()
print(f"\nMeilleure strategie : {best} (AUC={fusion_df.loc[best,'AUC']:.4f})")
print(f"Reference — Texte seul : {summarize(y, proba_camembert)['AUC']:.4f}")
print(f"Reference — Image seule : {summarize(y, proba_foundation)['AUC']:.4f}")

fusion_df.to_csv("/kaggle/working/fusion_5strategies_complet.csv")
print("\nSauvegarde -> fusion_5strategies_complet.csv")

In [ ]:
# On a deja les 6 seeds d'extraction BiomedCLIP -> reutiliser si stockees, sinon refaire vite
seeds_to_test = [1, 2, 3, 4, 42, 777]
aucs_fusion_multiseed = []
for s in seeds_to_test:
    embeds_s = extract_per_image_embeddings_foundation(
        patient_ids, cfg.IMAGE_DIR, model, preprocess, device,
        max_imgs_per_patient=30, seed=s
    )
    proba_img_s = run_loocv_slicelevel(patient_ids, y, embeds_s, seed=42, classifier="logreg", agg="mean")
    proba_fusion_s = fusion_moyenne_simple({"texte": proba_camembert, "image": proba_img_s})
    auc_s = roc_auc_score(y, proba_fusion_s)
    aucs_fusion_multiseed.append(auc_s)
    print(f"seed extraction={s}: AUC fusion={auc_s:.4f}")

print(f"\nFusion multi-seed : {np.mean(aucs_fusion_multiseed):.4f} +/- {np.std(aucs_fusion_multiseed):.4f}")

In [ ]:
seeds_to_test = [1, 2, 3, 4, 42, 777]
aucs_guided_multiseed = []
for s in seeds_to_test:
    embeds_s = extract_per_image_embeddings_foundation(
        patient_ids, cfg.IMAGE_DIR, model, preprocess, device,
        max_imgs_per_patient=30, seed=s
    )
    proba_img_s = run_loocv_slicelevel(patient_ids, y, embeds_s, seed=42, classifier="logreg", agg="mean")
    proba_guided_s = run_guided_fusion(proba_camembert, proba_img_s, y)
    auc_s = roc_auc_score(y, proba_guided_s)
    aucs_guided_multiseed.append(auc_s)
    print(f"seed extraction={s}: AUC fusion guidee={auc_s:.4f}")

print(f"\nFusion guidee multi-seed : {np.mean(aucs_guided_multiseed):.4f} +/- {np.std(aucs_guided_multiseed):.4f}")

In [ ]:
final_export = pd.DataFrame({
    "ID_Patient": patient_ids,
    "y_true": y,
    "proba_text_camembert": proba_camembert,
    "proba_image_biomedclip": proba_foundation,
    "proba_fusion_guided": proba_guided,
    "proba_fusion_moyenne_simple": fusion_moyenne_simple({"texte": proba_camembert, "image": proba_foundation}),
})
final_export.to_csv("/kaggle/working/final_probabilities_for_plots.csv", index=False)
print(final_export.head(10).to_string(index=False))

In [ ]:
for var in ["y", "patient_ids", "proba_camembert", "proba_foundation", "proba_guided"]:
    print(var, "->", var in dir())

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


def word_occlusion_importance(text, clf, scaler, embed_fn, baseline_proba):
    words = text.split()
    results = []
    for i in range(len(words)):
        text_occluded = " ".join(words[:i] + words[i+1:])
        emb = embed_fn([text_occluded])
        emb_scaled = scaler.transform(emb)
        proba_occluded = clf.predict_proba(emb_scaled)[0, 1]
        delta = baseline_proba - proba_occluded
        results.append({"mot": words[i], "delta_proba": delta})
    return pd.DataFrame(results).sort_values("delta_proba", key=abs, ascending=False)


def analyze_guided_fusion_weights(proba_text, proba_image, y, patient_ids):
    from scipy.stats import zscore
    text_confidence = 1 / (1 + np.exp(-zscore(proba_text)))
    agreement = 1 - np.abs(text_confidence - proba_image)
    w_text = agreement / (agreement + (1 - agreement) + 1e-9)

    df = pd.DataFrame({
        "ID_Patient": patient_ids,
        "y_true": y,
        "poids_texte": w_text,
        "poids_image": 1 - w_text,
        "proba_texte": proba_text,
        "proba_image": proba_image,
    })
    print(f"Poids moyen accorde au texte : {w_text.mean():.3f}")
    print(f"Poids moyen accorde a l'image : {(1-w_text).mean():.3f}")
    print(f"\nPatients ou le texte domine largement (poids>0.7) : {(w_text>0.7).sum()}")
    print(f"Patients ou l'image domine largement (poids<0.3) : {(w_text<0.3).sum()}")
    return df


def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted_transposed, label_1_count):
    m = label_1_count
    n = preds_sorted_transposed.shape[1] - m
    positive = preds_sorted_transposed[:, :m]
    negative = preds_sorted_transposed[:, m:]
    k = preds_sorted_transposed.shape[0]

    tx = np.empty([k, m])
    ty = np.empty([k, n])
    tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _compute_midrank(positive[r, :])
        ty[r, :] = _compute_midrank(negative[r, :])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])

    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1) / (2 * n)
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def delong_roc_test(y_true, proba_a, proba_b):
    order = np.argsort(-y_true)
    y_sorted = y_true[order]
    m = int(y_sorted.sum())
    preds = np.vstack([proba_a[order], proba_b[order]])
    aucs, delongcov = _fast_delong(preds, m)
    diff = aucs[0] - aucs[1]
    var = delongcov[0, 0] + delongcov[1, 1] - 2 * delongcov[0, 1]
    if var <= 1e-10:
        return diff, 1.0
    z = diff / np.sqrt(var)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return diff, p


print("Fonctions XAI/stats definies :", 'delong_roc_test' in dir())

# --- Test de DeLong : fusion guidee vs meilleure branche seule (image) ---
diff_auc, p_value = delong_roc_test(y, proba_guided, proba_foundation)
print(f"\nFusion guidee vs Image seule")
print(f"Difference d'AUC : {diff_auc:+.4f}")
print(f"p-value (DeLong) : {p_value:.4f}")
print(f"{'Significatif (p<0.05)' if p_value < 0.05 else 'NON significatif'}")

# --- Contribution de modalite ---
weights_df = analyze_guided_fusion_weights(proba_camembert, proba_foundation, y, patient_ids)
weights_df.to_csv("/kaggle/working/fusion_guided_weights_per_patient.csv", index=False)

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")  # ou colle le contenu du fichier
from xai_and_stats import (word_occlusion_importance, image_occlusion_saliency,
                            analyze_guided_fusion_weights, delong_roc_test)

# ============================================================
# 1. TEST STATISTIQUE FORMEL — fusion guidee vs meilleure branche seule
# ============================================================
diff_auc, p_value = delong_roc_test(y, proba_guided, proba_foundation)
print(f"Fusion guidee (0.914) vs Image seule (0.807, meilleure branche)")
print(f"Difference d'AUC : {diff_auc:+.4f}")
print(f"p-value (test de DeLong) : {p_value:.4f}")
print(f"{'Significatif (p<0.05)' if p_value < 0.05 else 'NON significatif malgre la difference visuelle'}")
print()

# ============================================================
# 2. CONTRIBUTION DE MODALITE (fusion guidee) — gratuit, deja calculable
# ============================================================
weights_df = analyze_guided_fusion_weights(proba_camembert, proba_foundation, y, patient_ids)
weights_df.to_csv("/kaggle/working/fusion_guided_weights_per_patient.csv", index=False)
print("\nExporte -> fusion_guided_weights_per_patient.csv")

# ============================================================
# 3. XAI TEXTE — exemple sur un patient (occlusion de mots)
# ============================================================
# Entrainer le classifieur final sur TOUT le dataset, a but interpretatif uniquement
# (pas pour rapporter une performance -- deja fait en LOO-CV plus haut)
scaler_text_final = StandardScaler()
X_text_final = scaler_text_final.fit_transform(camembert_embeddings)
clf_text_final = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)
clf_text_final.fit(X_text_final, y)

def embed_fn_camembert(texts):
    return extract_camembert_embeddings(pd.Series(texts), tokenizer, camembert, device)

patient_example_idx = 0  # a ajuster : choisir un patient CS bien predit
baseline = clf_text_final.predict_proba(X_text_final[patient_example_idx:patient_example_idx+1])[0,1]
importance_words = word_occlusion_importance(
    text_raw.iloc[patient_example_idx], clf_text_final, scaler_text_final,
    embed_fn_camembert, baseline
)
print(f"\nTop mots influents pour le patient {patient_ids[patient_example_idx]} :")
print(importance_words.head(15).to_string(index=False))
importance_words.to_csv("/kaggle/working/word_importance_example_patient.csv", index=False)

In [ ]:
# Reutilise les 6 seeds deja calcules pour le GAT texte+image
seeds_to_test = [1, 2, 3, 4, 42, 777]
all_probas_gat = []
for s in seeds_to_test:
    proba_s = run_loocv_gat(combined_features_v2, y, k=5, seed=s)
    all_probas_gat.append(proba_s)

proba_gat_ensemble = np.mean(all_probas_gat, axis=0)  # moyenne des PROBABILITES, pas des AUC
auc_ensemble = roc_auc_score(y, proba_gat_ensemble)
print(f"GAT ensemble (moyenne de 6 runs) : AUC={auc_ensemble:.4f}")
print(f"(vs moyenne des AUC individuels : {np.mean([roc_auc_score(y, p) for p in all_probas_gat]):.4f})")

In [ ]:
proba_gat_stable = run_loocv_gat(combined_features_v2, y, k=5, epochs=500, lr=0.001, seed=SEED)
print(f"GAT (epochs=500, lr=0.001) : AUC={roc_auc_score(y, proba_gat_stable):.4f}")

# Et un k different, puisque k=5 etait arbitraire depuis le debut
proba_gat_k10 = run_loocv_gat(combined_features_v2, y, k=10, seed=SEED)
print(f"GAT (k=10 voisins) : AUC={roc_auc_score(y, proba_gat_k10):.4f}")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re, glob, random, warnings, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from scipy import stats
from scipy.stats import zscore

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# ============================================================
# CONFIG
# ============================================================
class Config:
    EXCEL_PATH = Path("/kaggle/input/datasets/souaadrahmoun/datasetcac/ClasseurPFE1 (1).xlsx")
    IMAGE_DIR = Path("/kaggle/input/datasets/souaadrahmoun/datacacfinal/images_v2_final/data p_s")
    IMG_SIZE = 224
    MAX_IMGS_PER_PATIENT = 30

cfg = Config()
for p in [cfg.EXCEL_PATH, cfg.IMAGE_DIR]:
    print(p, "->", p.exists())

# ============================================================
# DONNEES + LABELS
# ============================================================
df = pd.read_excel(cfg.EXCEL_PATH, sheet_name="Feuil1")
for col in ["Origine_Tumeur", "ANNOTATION", "Rapport"]:
    df[col] = df[col].astype(str).fillna("").replace("nan", "")

y = (df["Origine_Tumeur"].str.strip() == "Secondaire").astype(int).values
patient_ids = df["ID_Patient"].values

CLASS_REVEALING_TERMS = [r"\bprimaire\b", r"\bsecondaire\b", r"m[ée]tastase\w*", r"m[ée]tastatique\w*"]
def mask_class_revealing_terms(text_series):
    out = text_series.astype(str)
    for pat in CLASS_REVEALING_TERMS:
        out = out.str.replace(pat, " CLASSE ", regex=True, flags=re.IGNORECASE)
    return out

text_raw = mask_class_revealing_terms(df["ANNOTATION"] + " " + df["Rapport"])
print(f"Patients : {len(y)} (CP={sum(y==0)}, CS={sum(y==1)})")

def summarize(y, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {"AUC": roc_auc_score(y, proba), "Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0, average="macro")}

# ============================================================
# CAMEMBERT (TEXTE)
# ============================================================
os.system("pip install transformers -q")
from transformers import AutoTokenizer, AutoModel

device = "cpu"
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
camembert = AutoModel.from_pretrained("camembert-base").to(device)
camembert.eval()

def extract_camembert_embeddings(text_series, tokenizer, model, device, max_length=256):
    embeddings = []
    with torch.no_grad():
        for text in text_series:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length, padding=True).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            summed = (outputs.last_hidden_state * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            embeddings.append((summed / counts).squeeze(0).cpu().numpy())
    return np.stack(embeddings)

camembert_embeddings = extract_camembert_embeddings(text_raw, tokenizer, camembert, device)
print("CamemBERT embeddings:", camembert_embeddings.shape)

def run_loocv_on_text_embeddings(embeddings, y, seed=SEED):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    for train_idx, test_idx in loo.split(embeddings):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(embeddings[train_idx])
        X_test = scaler.transform(embeddings[test_idx])
        clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0, random_state=seed)
        clf.fit(X_train, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test)[:, 1]
    return oof_proba

proba_camembert = run_loocv_on_text_embeddings(camembert_embeddings, y, seed=SEED)
print("Texte (CamemBERT):", summarize(y, proba_camembert))

# ============================================================
# BIOMEDCLIP (IMAGE) + CLASSIFICATION SLICE-LEVEL
# ============================================================
os.system("pip install open_clip_torch -q")
import open_clip
from PIL import Image

device_img = "cpu"

try:
    model, _, preprocess = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
    model_name_used = "BiomedCLIP"
except Exception as e:
    print(f"BiomedCLIP indisponible ({e}), repli CLIP general")
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    model_name_used = "CLIP general"

model.eval().to(device_img)
print("Modele image:", model_name_used)

def list_patient_images(patient_id, image_dir):
    pid_str = f"P{patient_id:02d}"
    files = []
    for cls in ["CP", "CS"]:
        cls_dir = image_dir / cls
        if cls_dir.exists():
            files += sorted(glob.glob(str(cls_dir / f"{pid_str} (*).jpg")))
            files += sorted(glob.glob(str(cls_dir / f"{pid_str} (*).png")))
    return files

def extract_per_image_embeddings_foundation(patient_ids, image_dir, model, preprocess, device,
                                             max_imgs_per_patient=30, seed=SEED):
    rng = random.Random(seed)
    embeddings_per_patient = {}
    for pid in patient_ids:
        files = list_patient_images(int(pid), image_dir)
        if len(files) > max_imgs_per_patient:
            files = rng.sample(files, max_imgs_per_patient)
        imgs = torch.stack([preprocess(Image.open(f).convert("RGB")) for f in files]).to(device)
        with torch.no_grad():
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        embeddings_per_patient[int(pid)] = feats.cpu().numpy()
        print(f"Patient {pid}: {len(files)} embeddings extraits")
    return embeddings_per_patient

def run_loocv_slicelevel(patient_ids, y, embeds_per_patient, seed=SEED, agg="mean"):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    for train_idx, test_idx in loo.split(patient_ids):
        train_pids = patient_ids[train_idx]
        test_pid = patient_ids[test_idx][0]
        X_train_slices, y_train_slices = [], []
        for pid, label in zip(train_pids, y[train_idx]):
            embeds = embeds_per_patient[int(pid)]
            X_train_slices.append(embeds)
            y_train_slices.append(np.full(len(embeds), label))
        X_train_slices = np.vstack(X_train_slices)
        y_train_slices = np.concatenate(y_train_slices)
        scaler = StandardScaler()
        X_train_slices = scaler.fit_transform(X_train_slices)
        clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0, random_state=seed)
        clf.fit(X_train_slices, y_train_slices)
        X_test_slices = scaler.transform(embeds_per_patient[int(test_pid)])
        proba_slices = clf.predict_proba(X_test_slices)[:, 1]
        oof_proba[test_idx] = np.mean(proba_slices) if agg == "mean" else np.median(proba_slices)
    return oof_proba

embeds_foundation = extract_per_image_embeddings_foundation(
    patient_ids, cfg.IMAGE_DIR, model, preprocess, device_img,
    max_imgs_per_patient=cfg.MAX_IMGS_PER_PATIENT, seed=SEED
)
proba_foundation = run_loocv_slicelevel(patient_ids, y, embeds_foundation, seed=SEED)
print("Image (BiomedCLIP):", summarize(y, proba_foundation))

# ============================================================
# FUSION GUIDEE + STATISTIQUE (DELONG)
# ============================================================
def fusion_moyenne_simple(branches):
    return np.stack(list(branches.values()), axis=1).mean(axis=1)

def run_guided_fusion(proba_text, proba_image, y):
    text_confidence = 1 / (1 + np.exp(-zscore(proba_text)))
    agreement = 1 - np.abs(text_confidence - proba_image)
    w_text = agreement / (agreement + (1 - agreement) + 1e-9)
    return w_text * proba_text + (1 - w_text) * proba_image

proba_guided = run_guided_fusion(proba_camembert, proba_foundation, y)
print("Fusion guidee:", summarize(y, proba_guided))

def _compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def _fast_delong(preds, m):
    n = preds.shape[1] - m
    positive, negative = preds[:, :m], preds[:, m:]
    k = preds.shape[0]
    tx, ty, tz = np.empty([k,m]), np.empty([k,n]), np.empty([k,m+n])
    for r in range(k):
        tx[r,:] = _compute_midrank(positive[r,:])
        ty[r,:] = _compute_midrank(negative[r,:])
        tz[r,:] = _compute_midrank(preds[r,:])
    aucs = tz[:,:m].sum(axis=1)/m/n - float(m+1)/(2*n)
    v01 = (tz[:,:m]-tx)/n; v10 = 1.0-(tz[:,m:]-ty)/m
    delongcov = np.cov(v01)/m + np.cov(v10)/n
    return aucs, delongcov

def delong_roc_test(y_true, proba_a, proba_b):
    order = np.argsort(-y_true)
    m = int(y_true[order].sum())
    preds = np.vstack([proba_a[order], proba_b[order]])
    aucs, cov = _fast_delong(preds, m)
    diff = aucs[0]-aucs[1]
    var = cov[0,0]+cov[1,1]-2*cov[0,1]
    if var <= 1e-10: return diff, 1.0
    z = diff/np.sqrt(var)
    return diff, 2*(1-stats.norm.cdf(abs(z)))

diff_auc, p_value = delong_roc_test(y, proba_guided, proba_foundation)
print(f"\nFusion guidee vs Image seule : diff={diff_auc:+.4f}, p={p_value:.4f}")
print("Significatif (p<0.05)" if p_value < 0.05 else "NON significatif")

# ============================================================
# EXPORT FINAL
# ============================================================
final_export = pd.DataFrame({
    "ID_Patient": patient_ids, "y_true": y,
    "proba_text_camembert": proba_camembert,
    "proba_image_biomedclip": proba_foundation,
    "proba_fusion_guided": proba_guided,
    "proba_fusion_moyenne_simple": fusion_moyenne_simple({"t": proba_camembert, "i": proba_foundation}),
})
final_export.to_csv("/kaggle/working/final_probabilities_for_plots.csv", index=False)
print("\n" + final_export.to_string(index=False))
print("\nExporte -> final_probabilities_for_plots.csv")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re, glob, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)

# ============================================================
# CONFIG + DONNEES
# ============================================================
class Config:
    EXCEL_PATH = Path("/kaggle/input/datasets/souaadrahmoun/datasetcac/ClasseurPFE1 (1).xlsx")
    IMAGE_DIR = Path("/kaggle/input/datasets/souaadrahmoun/datacacfinal/images_v2_final/data p_s")

cfg = Config()

df = pd.read_excel(cfg.EXCEL_PATH, sheet_name="Feuil1")
for col in ["Origine_Tumeur", "ANNOTATION", "Rapport"]:
    df[col] = df[col].astype(str).fillna("").replace("nan", "")

y = (df["Origine_Tumeur"].str.strip() == "Secondaire").astype(int).values
patient_ids = df["ID_Patient"].values

CLASS_REVEALING_TERMS = [r"\bprimaire\b", r"\bsecondaire\b", r"m[ée]tastase\w*", r"m[ée]tastatique\w*"]
def mask_class_revealing_terms(text_series):
    out = text_series.astype(str)
    for pat in CLASS_REVEALING_TERMS:
        out = out.str.replace(pat, " CLASSE ", regex=True, flags=re.IGNORECASE)
    return out

text_raw = mask_class_revealing_terms(df["ANNOTATION"] + " " + df["Rapport"])
print(f"Patients : {len(y)}")

# ============================================================
# CAMEMBERT — modele final (sur les 45 patients, but interpretatif uniquement)
# ============================================================
os.system("pip install transformers -q")
from transformers import AutoTokenizer, AutoModel

device = "cpu"
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
camembert = AutoModel.from_pretrained("camembert-base").to(device)
camembert.eval()

def extract_camembert_embeddings(text_series, tokenizer, model, device, max_length=256):
    embeddings = []
    with torch.no_grad():
        for text in text_series:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length, padding=True).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            summed = (outputs.last_hidden_state * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            embeddings.append((summed / counts).squeeze(0).cpu().numpy())
    return np.stack(embeddings)

camembert_embeddings = extract_camembert_embeddings(text_raw, tokenizer, camembert, device)

scaler_text = StandardScaler()
X_text_final = scaler_text.fit_transform(camembert_embeddings)
clf_text_final = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)
clf_text_final.fit(X_text_final, y)
print("Classifieur texte final entraine (interpretation uniquement)")

# ============================================================
# BIOMEDCLIP — modele final image
# ============================================================
os.system("pip install open_clip_torch -q")
import open_clip
from PIL import Image

device_img = "cpu"
try:
    model_img, _, preprocess = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    )
except Exception as e:
    print(f"BiomedCLIP indisponible ({e}), repli CLIP general")
    model_img, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
model_img.eval().to(device_img)

def list_patient_images(patient_id, image_dir):
    pid_str = f"P{patient_id:02d}"
    files = []
    for cls in ["CP", "CS"]:
        cls_dir = image_dir / cls
        if cls_dir.exists():
            files += sorted(glob.glob(str(cls_dir / f"{pid_str} (*).jpg")))
            files += sorted(glob.glob(str(cls_dir / f"{pid_str} (*).png")))
    return files

def embed_image(pil_img):
    with torch.no_grad():
        t = preprocess(pil_img).unsqueeze(0).to(device_img)
        feats = model_img.encode_image(t)
        feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().squeeze(0)

# Embeddings slice-level pour TOUS les patients (modele final, interpretation)
rng = random.Random(SEED)
X_img_slices, y_img_slices, patient_of_slice = [], [], []
for pid, label in zip(patient_ids, y):
    files = list_patient_images(int(pid), cfg.IMAGE_DIR)
    if len(files) > 30:
        files = rng.sample(files, 30)
    for f in files:
        X_img_slices.append(embed_image(Image.open(f).convert("RGB")))
        y_img_slices.append(label)
        patient_of_slice.append(pid)
X_img_slices = np.array(X_img_slices)
y_img_slices = np.array(y_img_slices)

scaler_img = StandardScaler()
X_img_scaled = scaler_img.fit_transform(X_img_slices)
clf_img_final = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)
clf_img_final.fit(X_img_scaled, y_img_slices)
print("Classifieur image final entraine (interpretation uniquement)")

# ============================================================
# XAI TEXTE — occlusion de mots (1 exemple CS bien predit)
# ============================================================
def word_occlusion_importance(text, clf, scaler, tokenizer, model, device, baseline_proba):
    words = text.split()
    results = []
    for i in range(len(words)):
        text_occluded = " ".join(words[:i] + words[i+1:])
        emb = extract_camembert_embeddings(pd.Series([text_occluded]), tokenizer, model, device)
        emb_scaled = scaler.transform(emb)
        proba_occluded = clf.predict_proba(emb_scaled)[0, 1]
        results.append({"mot": words[i], "delta_proba": baseline_proba - proba_occluded})
    return pd.DataFrame(results).sort_values("delta_proba", key=abs, ascending=False)

example_idx = np.where(y == 1)[0][0]  # premier patient CS
baseline_text = clf_text_final.predict_proba(X_text_final[example_idx:example_idx+1])[0, 1]
print(f"\nPatient {patient_ids[example_idx]} (CS) — proba de base: {baseline_text:.3f}")

importance_words = word_occlusion_importance(
    text_raw.iloc[example_idx], clf_text_final, scaler_text, tokenizer, camembert, device, baseline_text
)
print("Top 15 mots les plus influents :")
print(importance_words.head(15).to_string(index=False))
importance_words.to_csv("/kaggle/working/word_occlusion_example.csv", index=False)

# ============================================================
# XAI IMAGE — occlusion de patches (1 image du meme patient)
# ============================================================
def image_occlusion_saliency(image_pil, clf, scaler, baseline_proba, patch_size=32, stride=32):
    w, h = image_pil.size
    arr = np.array(image_pil)
    saliency = []
    for y0 in range(0, h, stride):
        for x0 in range(0, w, stride):
            occluded = arr.copy()
            occluded[y0:y0+patch_size, x0:x0+patch_size] = 127
            occ_img = Image.fromarray(occluded)
            emb = embed_image(occ_img).reshape(1, -1)
            emb_scaled = scaler.transform(emb)
            proba_occluded = clf.predict_proba(emb_scaled)[0, 1]
            saliency.append({"y": y0, "x": x0, "delta_proba": baseline_proba - proba_occluded})
    return pd.DataFrame(saliency)

files_example = list_patient_images(int(patient_ids[example_idx]), cfg.IMAGE_DIR)
if files_example:
    img_example = Image.open(files_example[0]).convert("RGB")
    baseline_img = clf_img_final.predict_proba(scaler_img.transform(embed_image(img_example).reshape(1,-1)))[0,1]
    print(f"\nImage occlusion sur {files_example[0]} — proba de base: {baseline_img:.3f}")
    saliency_df = image_occlusion_saliency(img_example, clf_img_final, scaler_img, baseline_img)
    saliency_df.to_csv("/kaggle/working/image_occlusion_example.csv", index=False)
    print("Zones les plus influentes (delta_proba le plus fort) :")
    print(saliency_df.sort_values("delta_proba", key=abs, ascending=False).head(5).to_string(index=False))

print("\nExporte -> word_occlusion_example.csv, image_occlusion_example.csv")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re
from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

SEED = 42
np.random.seed(SEED)

# ============================================================
# CONFIG + DONNEES
# ============================================================
EXCEL_PATH = Path("/kaggle/input/datasets/souaadrahmoun/datasetcac/ClasseurPFE1 (1).xlsx")

df = pd.read_excel(EXCEL_PATH, sheet_name="Feuil1")
for col in ["Origine_Tumeur", "Type_Général", "Sexe", "Localisation",
            "Métastase_Origine", "ANNOTATION", "Rapport"]:
    df[col] = df[col].astype(str).fillna("").replace("nan", "")

y = (df["Origine_Tumeur"].str.strip() == "Secondaire").astype(int).values
patient_ids = df["ID_Patient"].values
print(f"Patients : {len(y)} (CP={sum(y==0)}, CS={sum(y==1)})")

# ============================================================
# MASQUAGE CORRIGE — le bug etait ici :
# AVANT (bugue)  : r"\bsecondaire\b"   -> ne matche PAS "secondaires" (pluriel)
# APRES (corrige): r"secondaire\w*"    -> matche secondaire/secondaires/etc.
# + ajout de "primitif" qui manquait completement
# ============================================================
CLASS_REVEALING_TERMS_FIXED = [
    r"primaire\w*", r"secondaire\w*", r"m[ée]tastase\w*", r"m[ée]tastatique\w*",
    r"primitif\w*", r"primitiv\w*",
]

def mask_class_revealing_terms_fixed(text_series):
    out = text_series.astype(str)
    for pat in CLASS_REVEALING_TERMS_FIXED:
        out = out.str.replace(pat, " CLASSE ", regex=True, flags=re.IGNORECASE)
    return out

text_raw_fixed = mask_class_revealing_terms_fixed(df["ANNOTATION"] + " " + df["Rapport"])

# Verification : 0 terme residuel attendu
residual = text_raw_fixed.str.contains(r"secondaires?\b|primaires?\b|primitif\w*", case=False, regex=True).sum()
print(f"Termes revelateurs residuels apres correction : {residual}/45 (doit etre 0)")

# ============================================================
# Q1 — ABLATION TF-IDF (safe vs avec feature a risque), MASQUAGE CORRIGE
# ============================================================
def build_safe_manual_features(df):
    feat = pd.DataFrame(index=df.index)
    feat["Age"] = pd.to_numeric(df["Âge"] if "Âge" in df.columns else df["Age"], errors="coerce")
    feat["Sexe"] = LabelEncoder().fit_transform(df["Sexe"].astype(str))
    feat["Localisation"] = LabelEncoder().fit_transform(df["Localisation"].astype(str))
    annot = mask_class_revealing_terms_fixed(df["ANNOTATION"])
    nlp_patterns = {
        "nlp_necrose": r"n[ée]cros", "nlp_oedeme": r"[oœ]d[eè]me",
        "nlp_effet_masse": r"effet de masse", "nlp_rehaussement": r"rehaussement",
        "nlp_saignement": r"saignement|h[ée]morrag", "nlp_multiple": r"multiple|plusieurs",
        "nlp_unique": r"\bunique\b", "nlp_engagement": r"engagement|sous-falc",
        "nlp_spectro": r"spectroscop", "nlp_annulaire": r"annulaire",
        "nlp_corps_calleux": r"corps calleux", "nlp_jonction_cortico": r"jonction cortico|cortico-sous-cortical",
        "nlp_infiltrant": r"infiltrant\w*",
    }
    annot_lower = annot.str.lower()
    for name, pat in nlp_patterns.items():
        feat[name] = annot_lower.str.contains(pat, na=False).astype(int)
    return feat

def build_leaky_features(df):
    feat = pd.DataFrame(index=df.index)
    meta_vide = ["", "nan", "none", "non applicable", "non précisée", "-", "n/a"]
    feat["a_metastase"] = (~df["Métastase_Origine"].astype(str).str.strip().str.lower().isin(meta_vide)).astype(int)
    return feat

class DenseTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X.toarray() if hasattr(X, "toarray") else X

def run_loocv_text_model(manual_feat, text_raw, y, k_best=20, seed=SEED):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    manual_arr = manual_feat.values.astype(float)
    text_arr = text_raw.values
    for train_idx, test_idx in loo.split(manual_arr):
        try:
            tfidf = TfidfVectorizer(max_features=150, ngram_range=(1, 2), min_df=2, max_df=0.90, sublinear_tf=True)
            X_tfidf_train = tfidf.fit_transform(text_arr[train_idx])
        except ValueError:
            tfidf = TfidfVectorizer(max_features=150, ngram_range=(1, 1), min_df=1, max_df=1.0, sublinear_tf=True)
            X_tfidf_train = tfidf.fit_transform(text_arr[train_idx])
        X_tfidf_test = tfidf.transform(text_arr[test_idx])
        dense = DenseTransformer()
        X_tfidf_train = dense.transform(X_tfidf_train)
        X_tfidf_test = dense.transform(X_tfidf_test)
        med = np.nanmedian(manual_arr[train_idx], axis=0)
        Xm_train = np.where(np.isnan(manual_arr[train_idx]), med, manual_arr[train_idx])
        Xm_test = np.where(np.isnan(manual_arr[test_idx]), med, manual_arr[test_idx])
        X_train_full = np.hstack([Xm_train, X_tfidf_train])
        X_test_full = np.hstack([Xm_test, X_tfidf_test])
        scaler = StandardScaler()
        X_train_full = scaler.fit_transform(X_train_full)
        X_test_full = scaler.transform(X_test_full)
        mi_scorer = partial(mutual_info_classif, random_state=seed)
        k = min(k_best, X_train_full.shape[1])
        selector = SelectKBest(mi_scorer, k=k)
        X_train_sel = selector.fit_transform(X_train_full, y[train_idx])
        X_test_sel = selector.transform(X_test_full)
        clf = RandomForestClassifier(n_estimators=300, max_depth=5, class_weight="balanced", random_state=seed)
        clf.fit(X_train_sel, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test_sel)[:, 1]
    return oof_proba

def summarize(y, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {"AUC": roc_auc_score(y, proba), "Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0, average="macro")}

safe_feat = build_safe_manual_features(df)
leaky_feat = build_leaky_features(df)

print("\n" + "="*70)
print("Q1 REFAIT (masquage corrige) — Texte SANS feature a risque")
print("="*70)
proba_safe_fixed = run_loocv_text_model(safe_feat, text_raw_fixed, y, seed=SEED)
print(summarize(y, proba_safe_fixed))

print("\n" + "="*70)
print("Q1 REFAIT (masquage corrige) — Texte AVEC feature a risque")
print("="*70)
combined_feat = pd.concat([safe_feat, leaky_feat], axis=1)
proba_leaky_fixed = run_loocv_text_model(combined_feat, text_raw_fixed, y, seed=SEED)
print(summarize(y, proba_leaky_fixed))

delta = summarize(y, proba_leaky_fixed)["AUC"] - summarize(y, proba_safe_fixed)["AUC"]
print(f"\nDelta AUC (masquage corrige) : {delta:+.4f}")

# ============================================================
# CAMEMBERT REFAIT (masquage corrige)
# ============================================================
os.system("pip install transformers -q")
from transformers import AutoTokenizer, AutoModel

device = "cpu"
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
camembert = AutoModel.from_pretrained("camembert-base").to(device)
camembert.eval()

def extract_camembert_embeddings(text_series, tokenizer, model, device, max_length=256):
    embeddings = []
    with torch.no_grad():
        for text in text_series:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length, padding=True).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            summed = (outputs.last_hidden_state * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            embeddings.append((summed / counts).squeeze(0).cpu().numpy())
    return np.stack(embeddings)

camembert_embeddings_fixed = extract_camembert_embeddings(text_raw_fixed, tokenizer, camembert, device)

def run_loocv_on_text_embeddings(embeddings, y, seed=SEED):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    for train_idx, test_idx in loo.split(embeddings):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(embeddings[train_idx])
        X_test = scaler.transform(embeddings[test_idx])
        clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0, random_state=seed)
        clf.fit(X_train, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test)[:, 1]
    return oof_proba

proba_camembert_fixed = run_loocv_on_text_embeddings(camembert_embeddings_fixed, y, seed=SEED)

print("\n" + "="*70)
print("CAMEMBERT REFAIT (masquage corrige)")
print("="*70)
print("AVANT (masquage bugue)  : AUC = 0.7626  (resultat original du papier)")
print("APRES (masquage corrige):", summarize(y, proba_camembert_fixed))

# ============================================================
# EXPORT
# ============================================================
pd.DataFrame({
    "ID_Patient": patient_ids, "y_true": y,
    "proba_text_safe_fixed": proba_safe_fixed,
    "proba_text_leaky_fixed": proba_leaky_fixed,
    "proba_camembert_fixed": proba_camembert_fixed,
}).to_csv("/kaggle/working/results_masking_fixed.csv", index=False)
print("\nExporte -> results_masking_fixed.csv")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re
from functools import partial
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

SEED = 42
np.random.seed(SEED)

# ============================================================
# CONFIG + DONNEES
# ============================================================
EXCEL_PATH = Path("/kaggle/input/datasets/souaadrahmoun/datasetcac/ClasseurPFE1 (1).xlsx")

df = pd.read_excel(EXCEL_PATH, sheet_name="Feuil1")
for col in ["Origine_Tumeur", "Sexe", "Localisation", "Métastase_Origine", "ANNOTATION", "Rapport"]:
    df[col] = df[col].astype(str).fillna("").replace("nan", "")

y = (df["Origine_Tumeur"].str.strip() == "Secondaire").astype(int).values
patient_ids = df["ID_Patient"].values
print(f"Patients : {len(y)} (CP={sum(y==0)}, CS={sum(y==1)})")

# ============================================================
# MASQUAGE CORRIGE (regex fixe : \w* au lieu de rien, + primitif ajoute)
# ============================================================
CLASS_REVEALING_TERMS_FIXED = [
    r"primaire\w*", r"secondaire\w*", r"m[ée]tastase\w*", r"m[ée]tastatique\w*",
    r"primitif\w*", r"primitiv\w*",
]

def mask_class_revealing_terms_fixed(text_series):
    out = text_series.astype(str)
    for pat in CLASS_REVEALING_TERMS_FIXED:
        out = out.str.replace(pat, " CLASSE ", regex=True, flags=re.IGNORECASE)
    return out

text_raw_fixed = mask_class_revealing_terms_fixed(df["ANNOTATION"] + " " + df["Rapport"])
residual = text_raw_fixed.str.contains(r"secondaires?\b|primaires?\b|primitif\w*", case=False, regex=True).sum()
print(f"Verification masquage : {residual}/45 termes residuels (doit etre 0)")

# ============================================================
# FEATURES (identiques a avant)
# ============================================================
def build_safe_manual_features(df):
    feat = pd.DataFrame(index=df.index)
    feat["Age"] = pd.to_numeric(df["Âge"] if "Âge" in df.columns else df["Age"], errors="coerce")
    feat["Sexe"] = LabelEncoder().fit_transform(df["Sexe"].astype(str))
    feat["Localisation"] = LabelEncoder().fit_transform(df["Localisation"].astype(str))
    annot = mask_class_revealing_terms_fixed(df["ANNOTATION"])
    nlp_patterns = {
        "nlp_necrose": r"n[ée]cros", "nlp_oedeme": r"[oœ]d[eè]me",
        "nlp_effet_masse": r"effet de masse", "nlp_rehaussement": r"rehaussement",
        "nlp_saignement": r"saignement|h[ée]morrag", "nlp_multiple": r"multiple|plusieurs",
        "nlp_unique": r"\bunique\b", "nlp_engagement": r"engagement|sous-falc",
        "nlp_spectro": r"spectroscop", "nlp_annulaire": r"annulaire",
        "nlp_corps_calleux": r"corps calleux", "nlp_jonction_cortico": r"jonction cortico|cortico-sous-cortical",
        "nlp_infiltrant": r"infiltrant\w*",
    }
    annot_lower = annot.str.lower()
    for name, pat in nlp_patterns.items():
        feat[name] = annot_lower.str.contains(pat, na=False).astype(int)
    return feat

def build_leaky_features(df):
    feat = pd.DataFrame(index=df.index)
    meta_vide = ["", "nan", "none", "non applicable", "non précisée", "-", "n/a"]
    feat["a_metastase"] = (~df["Métastase_Origine"].astype(str).str.strip().str.lower().isin(meta_vide)).astype(int)
    return feat

class DenseTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X.toarray() if hasattr(X, "toarray") else X

def run_loocv_text_model(manual_feat, text_raw, y, k_best=20, seed=SEED):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    manual_arr = manual_feat.values.astype(float)
    text_arr = text_raw.values
    for train_idx, test_idx in loo.split(manual_arr):
        try:
            tfidf = TfidfVectorizer(max_features=150, ngram_range=(1, 2), min_df=2, max_df=0.90, sublinear_tf=True)
            X_tfidf_train = tfidf.fit_transform(text_arr[train_idx])
        except ValueError:
            tfidf = TfidfVectorizer(max_features=150, ngram_range=(1, 1), min_df=1, max_df=1.0, sublinear_tf=True)
            X_tfidf_train = tfidf.fit_transform(text_arr[train_idx])
        X_tfidf_test = tfidf.transform(text_arr[test_idx])
        dense = DenseTransformer()
        X_tfidf_train = dense.transform(X_tfidf_train)
        X_tfidf_test = dense.transform(X_tfidf_test)
        med = np.nanmedian(manual_arr[train_idx], axis=0)
        Xm_train = np.where(np.isnan(manual_arr[train_idx]), med, manual_arr[train_idx])
        Xm_test = np.where(np.isnan(manual_arr[test_idx]), med, manual_arr[test_idx])
        X_train_full = np.hstack([Xm_train, X_tfidf_train])
        X_test_full = np.hstack([Xm_test, X_tfidf_test])
        scaler = StandardScaler()
        X_train_full = scaler.fit_transform(X_train_full)
        X_test_full = scaler.transform(X_test_full)
        mi_scorer = partial(mutual_info_classif, random_state=seed)
        k = min(k_best, X_train_full.shape[1])
        selector = SelectKBest(mi_scorer, k=k)
        X_train_sel = selector.fit_transform(X_train_full, y[train_idx])
        X_test_sel = selector.transform(X_test_full)
        clf = RandomForestClassifier(n_estimators=300, max_depth=5, class_weight="balanced", random_state=seed)
        clf.fit(X_train_sel, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test_sel)[:, 1]
    return oof_proba

def run_loocv_on_text_embeddings(embeddings, y, seed=SEED):
    n = len(y)
    loo = LeaveOneOut()
    oof_proba = np.zeros(n)
    for train_idx, test_idx in loo.split(embeddings):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(embeddings[train_idx])
        X_test = scaler.transform(embeddings[test_idx])
        clf = LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0, random_state=seed)
        clf.fit(X_train, y[train_idx])
        oof_proba[test_idx] = clf.predict_proba(X_test)[:, 1]
    return oof_proba

def summarize(y, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {"AUC": roc_auc_score(y, proba), "Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0, average="macro")}

safe_feat = build_safe_manual_features(df)
leaky_feat = build_leaky_features(df)
combined_feat = pd.concat([safe_feat, leaky_feat], axis=1)

# ============================================================
# CAMEMBERT — embeddings extraits UNE FOIS (deterministe, pas besoin de multi-seed sur l'extraction)
# ============================================================
os.system("pip install transformers -q")
from transformers import AutoTokenizer, AutoModel

device = "cpu"
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
camembert = AutoModel.from_pretrained("camembert-base").to(device)
camembert.eval()

def extract_camembert_embeddings(text_series, tokenizer, model, device, max_length=256):
    embeddings = []
    with torch.no_grad():
        for text in text_series:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length, padding=True).to(device)
            outputs = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            summed = (outputs.last_hidden_state * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            embeddings.append((summed / counts).squeeze(0).cpu().numpy())
    return np.stack(embeddings)

camembert_embeddings_fixed = extract_camembert_embeddings(text_raw_fixed, tokenizer, camembert, device)
print("Embeddings CamemBERT extraits (deterministe, une seule fois)")

# ============================================================
# MULTI-SEED (6 seeds), masquage corrige
# ============================================================
seeds_to_test = [1, 2, 3, 4, 42, 777]
results_multiseed = []

for s in seeds_to_test:
    print(f"\n{'='*20} SEED {s} {'='*20}")

    proba_safe_s = run_loocv_text_model(safe_feat, text_raw_fixed, y, seed=s)
    proba_leaky_s = run_loocv_text_model(combined_feat, text_raw_fixed, y, seed=s)
    proba_camembert_s = run_loocv_on_text_embeddings(camembert_embeddings_fixed, y, seed=s)

    auc_safe = summarize(y, proba_safe_s)["AUC"]
    auc_leaky = summarize(y, proba_leaky_s)["AUC"]
    auc_camembert = summarize(y, proba_camembert_s)["AUC"]

    print(f"Safe (sans feature a risque)   : AUC={auc_safe:.4f}")
    print(f"Leaky (avec feature a risque)  : AUC={auc_leaky:.4f}")
    print(f"CamemBERT                      : AUC={auc_camembert:.4f}")

    results_multiseed.append({
        "seed": s, "AUC_safe": auc_safe, "AUC_leaky": auc_leaky, "AUC_camembert": auc_camembert
    })

# ============================================================
# RESUME
# ============================================================
results_df = pd.DataFrame(results_multiseed)
print("\n" + "="*70)
print("RESUME MULTI-SEED (masquage corrige)")
print("="*70)
summary = results_df.drop(columns="seed").agg(["mean", "std", "min", "max"])
print(summary.to_string())

delta_per_seed = results_df["AUC_leaky"] - results_df["AUC_safe"]
print(f"\nDelta AUC (fuite) par seed : {delta_per_seed.tolist()}")
print(f"Delta AUC moyen : {delta_per_seed.mean():.4f} +/- {delta_per_seed.std():.4f}")

results_df.to_csv("/kaggle/working/multiseed_corrected_masking.csv", index=False)
print("\nExporte -> multiseed_corrected_masking.csv")